# Sci-epo × MiniMax H3 worker · 渲染工作机

Renders the jobs Sci-epo plans from an adopted shot list: one job is one clip in one voice language, with its H3 prompt, frames, size, seed and reference files. Videos and `results.json` go to Google Drive. Nothing but job status goes back to Sci-epo.

Pick **one** connection mode · 三种连接方式选一种：

- **A · Job package 任务包**: download the package on the 制作 tab, put it in Drive, render, upload `results.json` back.
- **B · Pull 领取**: Sci-epo is online. Create a worker token on the 制作 tab and store it in Colab Secrets as `SCIEPO_WORKER_TOKEN`.
- **C · Serve 推送**: Sci-epo runs on your own computer. This notebook opens a tunnel; paste its URL and key into the batch on the 制作 tab.

Use a GPU runtime (H3 is large; an A100 or better is recommended). Guide: [docs/production-bridge.md](https://github.com/zhang001ziyu/sci-epo/blob/main/docs/production-bridge.md).

In [ ]:
#@title 1 · Settings and Google Drive · 设置并挂载云端硬盘
DRIVE_DIR = '/content/drive/MyDrive/SciEpo'  #@param {type:"string"}
COMFY_DIR = '/content/ComfyUI'  #@param {type:"string"}
COMFY_URL = 'http://127.0.0.1:8188'  #@param {type:"string"}
import os
from google.colab import drive
drive.mount('/content/drive')
for sub in ('packages', 'renders', 'refs', 'workflows'):
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)
T2V_WORKFLOW = f'{DRIVE_DIR}/workflows/h3_t2v_api.json'
R2V_WORKFLOW = f'{DRIVE_DIR}/workflows/h3_r2v_api.json'
REFS = f'{DRIVE_DIR}/refs'
print('job packages  ->', f'{DRIVE_DIR}/packages')
print('reference images and voice samples ->', REFS)
print('workflows     ->', f'{DRIVE_DIR}/workflows')
print('videos        ->', f'{DRIVE_DIR}/renders')

In [ ]:
#@title 2 · Optional: install ComfyUI and the MiniMax H3 models · 可选：安装 ComfyUI 与 H3 模型
#@markdown Skip this if your Colab already runs H3 in ComfyUI. The files are large (tens of GB) and come from the official Comfy-Org/MiniMax-H3 repository on Hugging Face.
INSTALL = False  #@param {type:"boolean"}
WITH_R2V = True  #@param {type:"boolean"}
if INSTALL:
    if not os.path.isdir(COMFY_DIR):
        !git clone --depth 1 https://github.com/Comfy-Org/ComfyUI {COMFY_DIR}
    !pip install -q -r {COMFY_DIR}/requirements.txt
    hub = 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main'
    files = [('diffusion_models', f'{hub}/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors'),
             ('text_encoders', f'{hub}/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'),
             ('vae', f'{hub}/vae/minimax_h3_video_vae_int8_convrot.safetensors'),
             ('vae', f'{hub}/vae/minimax_h3_audio_vae_fp32.safetensors'),
             ('loras', 'https://huggingface.co/lightx2v/Minimax-h3-Turbo/resolve/main/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors')]
    if WITH_R2V:
        files += [('diffusion_models', f'{hub}/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors'),
                  ('loras', f'{hub}/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors')]
    for folder, url in files:
        !wget -q -nc -P {COMFY_DIR}/models/{folder} {url}
    print('installed; load the MiniMax H3 T2V and R2V templates in ComfyUI and export them with Workflow > Export (API)')

In [ ]:
#@title 3 · Start ComfyUI in the background · 后台启动 ComfyUI
#@markdown Skip the start if ComfyUI already runs on the port above. This cell then only checks that it answers.
START_COMFY = False  #@param {type:"boolean"}
import subprocess, time, urllib.request
if START_COMFY:
    comfy_log = open('/content/comfyui.log', 'w')
    subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1', '--port', COMFY_URL.rsplit(':', 1)[1]], cwd=COMFY_DIR, stdout=comfy_log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen(COMFY_URL + '/system_stats', timeout=3)
        print('ComfyUI answers at', COMFY_URL)
        break
    except Exception:
        time.sleep(5)
else:
    print('ComfyUI does not answer at', COMFY_URL, '- see /content/comfyui.log')

In [ ]:
#@title 4 · Get the Sci-epo worker · 下载工作机程序
!wget -q -O /content/sciepo_worker.py https://raw.githubusercontent.com/zhang001ziyu/sci-epo/main/worker/sciepo_worker.py
!python /content/sciepo_worker.py --help

In [ ]:
#@title 5 · Prepare your workflows once · 准备工作流（只需一次）
#@markdown In ComfyUI, open the MiniMax H3 T2V template (and R2V if you use reference pictures), then Workflow > Export (API). Save the files to the workflows folder with these names. This cell puts {{PROMPT}}, {{WIDTH}}, {{FRAMES}}, {{SEED}}, {{IMAGE_1}} and the other placeholders in; read every listed change.
T2V_EXPORT = f'{DRIVE_DIR}/workflows/h3_t2v_export.json'  #@param {type:"string"}
R2V_EXPORT = f'{DRIVE_DIR}/workflows/h3_r2v_export.json'  #@param {type:"string"}
if os.path.exists(T2V_EXPORT):
    !python /content/sciepo_worker.py prepare-workflow --input "{T2V_EXPORT}" --output "{T2V_WORKFLOW}"
else:
    print('no T2V export at', T2V_EXPORT)
if os.path.exists(R2V_EXPORT):
    !python /content/sciepo_worker.py prepare-workflow --input "{R2V_EXPORT}" --output "{R2V_WORKFLOW}"
else:
    print('no R2V export; jobs with reference pictures will fail until you add one')

def backend_args(backend='comfyui'):
    args = ['--backend', backend, '--comfy-url', COMFY_URL, '--refs', REFS]
    if os.path.exists(T2V_WORKFLOW): args += ['--workflow', T2V_WORKFLOW]
    if os.path.exists(R2V_WORKFLOW): args += ['--r2v-workflow', R2V_WORKFLOW]
    return args

In [ ]:
#@title A · Render a job package · 渲染任务包
PACKAGE = f'{DRIVE_DIR}/packages/sciepo-render-BATCH.json'  #@param {type:"string"}
DRY_RUN = False  #@param {type:"boolean"}
import shlex
out = f"{DRIVE_DIR}/renders/{os.path.basename(PACKAGE).removesuffix('.json')}"
cmd = ['python', '/content/sciepo_worker.py', 'package', '--package', PACKAGE, '--out', out] + backend_args('dry-run' if DRY_RUN else 'comfyui')
!{shlex.join(cmd)}
print('Upload', f'{out}/results.json', 'on the 制作 tab (上传结果).')

In [ ]:
#@title B · Pull jobs from an online Sci-epo · 从在线的 Sci-epo 领取任务
#@markdown Store the worker token in Colab Secrets (the key icon) as SCIEPO_WORKER_TOKEN and allow this notebook to read it.
SERVER = 'https://studio.example.com'  #@param {type:"string"}
KEEP_WAITING = True  #@param {type:"boolean"}
import shlex
from google.colab import userdata
os.environ['SCIEPO_WORKER_TOKEN'] = userdata.get('SCIEPO_WORKER_TOKEN')
cmd = ['python', '/content/sciepo_worker.py', 'pull', '--server', SERVER, '--out', f'{DRIVE_DIR}/renders'] + backend_args() + (['--wait'] if KEEP_WAITING else [])
!{shlex.join(cmd)}

In [ ]:
#@title C · Serve for a Sci-epo on your computer · 给本机的 Sci-epo 开推送入口
#@markdown Starts the worker on port 8765 and a Cloudflare quick tunnel. Paste the printed URL and key into the batch on the 制作 tab, then 发送到工作机; after rendering, 同步结果.
import re, secrets, subprocess, time
if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
KEY = secrets.token_urlsafe(24)
worker_log = open('/content/sciepo-worker.log', 'w')
worker = subprocess.Popen(['python', '/content/sciepo_worker.py', 'serve', '--port', '8765', '--key', KEY, '--out', f'{DRIVE_DIR}/renders'] + backend_args(), stdout=worker_log, stderr=subprocess.STDOUT)
tunnel_log = open('/content/cloudflared.log', 'w')
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8765'], stdout=tunnel_log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
    if found:
        url = found.group(0)
        break
    time.sleep(2)
print('Worker URL 工作机地址:', url or 'not ready, see /content/cloudflared.log')
print('Worker key 工作机密钥:', KEY)
print('Logs: /content/sciepo-worker.log. Keep this runtime open while it renders.')

In [ ]:
#@title Worker log (mode C) · 查看工作机日志
!tail -n 40 /content/sciepo-worker.log